<a href="https://colab.research.google.com/github/nuruliman-web/APUPPT/blob/main/DTTOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
# MATCHING NASABAH VS TERORIS - FUZZY 95%
# AUTO DETECT KOLOM - TANPA INPUT
# ================================================================

from google.colab import files
import pandas as pd
import numpy as np
from difflib import SequenceMatcher
import warnings
import re
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
warnings.filterwarnings('ignore')

print("="*70)
print("🔍 SISTEM MATCHING NASABAH VS DATA TERORIS (FUZZY 95%)")
print("="*70)
print("\n📌 SISTEM AKAN MENDETEKSI KOLOM SECARA OTOMATIS")

# ================================================================
# STEP 1: UPLOAD FILE
# ================================================================

print("\n📤 STEP 1: UPLOAD FILE")
print("-"*70)

print("\n📤 UPLOAD FILE DATABASE NASABAH")
print("   (Harus ada kolom: CIF, Nama, Tgl Lahir, NIK)")
uploaded_nasabah = files.upload()

print("\n📤 UPLOAD FILE DATABASE TERORIS")
print("   (Harus ada kolom: Nama, Nama1-Nama31, Deskripsi, Terduga, Kode Densus, Tempat Lahir, Tanggal Lahir, WN/Asal Negara, Alamat)")
uploaded_teroris = files.upload()

# Ambil nama file
file_nasabah = list(uploaded_nasabah.keys())[0]
file_teroris = list(uploaded_teroris.keys())[0]

print(f"\n✅ File nasabah: {file_nasabah}")
print(f"✅ File teroris: {file_teroris}")

# ================================================================
# STEP 2: LOAD DATA (AUTO DETECT HEADER)
# ================================================================

print("\n📂 STEP 2: MEMBACA DATA")
print("-"*70)

def load_excel_auto(filename):
    """Load Excel dengan auto detect header"""
    try:
        # Coba baca dengan header di baris pertama
        df = pd.read_excel(filename, header=0)
        return df
    except:
        # Jika gagal, coba tanpa header
        df = pd.read_excel(filename, header=None)
        # Coba gunakan baris pertama sebagai header
        df.columns = df.iloc[0].astype(str)
        df = df[1:].reset_index(drop=True)
        return df

df_nasabah = load_excel_auto(file_nasabah)
df_teroris = load_excel_auto(file_teroris)

print(f"\n✅ Data Nasabah: {df_nasabah.shape[0]} baris, {df_nasabah.shape[1]} kolom")
print(f"✅ Data Teroris: {df_teroris.shape[0]} baris, {df_teroris.shape[1]} kolom")

# ================================================================
# STEP 3: AUTO DETECT KOLOM
# ================================================================

print("\n🔍 STEP 3: AUTO DETECT KOLOM")
print("-"*70)

# Auto detect kolom nasabah
print("\n📌 DETEKSI KOLOM NASABAH:")

# Cari kolom CIF
col_cif = None
for col in df_nasabah.columns:
    col_str = str(col).strip().upper()
    if 'CIF' in col_str or 'NO' in col_str or 'ID' in col_str:
        col_cif = col
        break

# Cari kolom Nama
col_nama_nasabah = None
for col in df_nasabah.columns:
    col_str = str(col).strip().upper()
    if 'NAMA' in col_str and 'NASABAH' not in col_str:
        col_nama_nasabah = col
        break

# Cari kolom Tgl Lahir
col_tgl_lahir = None
for col in df_nasabah.columns:
    col_str = str(col).strip().upper()
    if 'TGL' in col_str or 'TANGGAL' in col_str or 'LAHIR' in col_str:
        col_tgl_lahir = col
        break

# Cari kolom NIK
col_nik = None
for col in df_nasabah.columns:
    col_str = str(col).strip().upper()
    if 'NIK' in col_str or 'KTP' in col_str or 'IDENTITAS' in col_str:
        col_nik = col
        break

# Auto detect kolom teroris
print("\n📌 DETEKSI KOLOM TERORIS:")

# Cari kolom Nama Asli
col_nama_teroris = None
for col in df_teroris.columns:
    col_str = str(col).strip()
    if col_str == 'Nama' and 'Nama1' not in col_str:
        col_nama_teroris = col
        break

# Ambil semua kolom alias (Nama1 - Nama31)
alias_columns = []
for col in df_teroris.columns:
    col_str = str(col).strip()
    if col_str.startswith('Nama') and col_str != 'Nama':
        try:
            num = int(col_str.replace('Nama', ''))
            if 1 <= num <= 31:
                alias_columns.append(col)
        except:
            pass

# Sort alias
alias_columns = sorted(alias_columns, key=lambda x: int(str(x).replace('Nama', '')) if str(x).replace('Nama', '').isdigit() else 0)

# Tampilkan hasil deteksi
print("\n✅ KOLOM YANG TERDETEKSI:")
print(f"   • CIF Nasabah          : {col_cif if col_cif else 'TIDAK DITEMUKAN'}")
print(f"   • Nama Nasabah         : {col_nama_nasabah if col_nama_nasabah else 'TIDAK DITEMUKAN'}")
print(f"   • Tgl Lahir Nasabah    : {col_tgl_lahir if col_tgl_lahir else 'TIDAK DITEMUKAN'}")
print(f"   • NIK Nasabah          : {col_nik if col_nik else 'TIDAK DITEMUKAN'}")
print(f"   • Nama Asli Teroris    : {col_nama_teroris if col_nama_teroris else 'TIDAK DITEMUKAN'}")
print(f"   • Kolom Alias Teroris  : {len(alias_columns)} kolom ditemukan")

if len(alias_columns) > 0:
    print(f"\n   📋 ALIAS YANG DITEMUKAN:")
    for i, col in enumerate(alias_columns[:10]):
        print(f"      {i+1}. {col}")
    if len(alias_columns) > 10:
        print(f"      ... dan {len(alias_columns)-10} kolom lainnya")

# ================================================================
# STEP 4: FUNGSI MATCHING
# ================================================================

print("\n⚙️ STEP 4: PROSES MATCHING")
print("-"*70)

def clean_name(name):
    """Membersihkan nama dari gelar dan tanda baca"""
    if pd.isna(name):
        return ""
    name = str(name).upper().strip()
    # Hapus gelar
    gelar = ['DR.', 'DR', 'H.', 'H', 'IR.', 'IR', 'PROF.', 'PROF',
             'S.KOM', 'S.H', 'M.M', 'M.KOM', 'S.E', 'BIN', 'BSC', 'MBA',
             'PH.D', 'PH.D.', 'M.SC', 'M.SC.', 'Drs.', 'Drs', 'Dra.', 'Dra']
    for g in gelar:
        name = name.replace(g, '')
    # Hapus tanda baca
    name = re.sub(r'[^\w\s]', ' ', name)
    # Hapus multiple spasi
    name = ' '.join(name.split())
    return name

def similarity(a, b):
    """Menghitung tingkat kemiripan 2 string"""
    if not a or not b:
        return 0
    return SequenceMatcher(None, a, b).ratio()

# ================================================================
# STEP 5: PROSES MATCHING UTAMA
# ================================================================

print("\n🔄 MEMPROSES MATCHING (95% threshold)...")

# Threshold
THRESHOLD = 0.95

# Siapkan data teroris untuk matching
teroris_list = []
print("\n📊 MEMPROSES DATA TERORIS...")
for idx, row in df_teroris.iterrows():
    if col_nama_teroris is None or pd.isna(row[col_nama_teroris]):
        continue

    nama_asli = row[col_nama_teroris]
    all_names = [nama_asli]

    # Ambil semua alias
    for alias_col in alias_columns:
        if alias_col in row and pd.notna(row[alias_col]):
            alias = str(row[alias_col]).strip()
            if alias and alias != 'nan':
                all_names.append(alias)

    teroris_list.append({
        'row_data': row,
        'nama_asli': nama_asli,
        'all_names': all_names,
        'all_names_clean': [clean_name(n) for n in all_names if clean_name(n)]
    })

print(f"✅ Total data teroris: {len(teroris_list)}")

# Proses setiap nasabah
results = []
false_positives = []
false_negatives = []

print("\n🔍 MENCARI MATCHING...")
total_nasabah = len(df_nasabah)

for idx, nasabah_row in df_nasabah.iterrows():
    if col_nama_nasabah is None or pd.isna(nasabah_row[col_nama_nasabah]):
        continue

    nama_nasabah = nasabah_row[col_nama_nasabah]
    nama_clean_nasabah = clean_name(nama_nasabah)

    if not nama_clean_nasabah:
        continue

    match_found = False
    matched_aliases = []
    matched_teroris_data = None
    best_score = 0

    for teroris in teroris_list:
        matched_alias_for_this_teroris = []
        is_match = False

        for i, nama_teroris in enumerate(teroris['all_names_clean']):
            if not nama_teroris:
                continue
            sim = similarity(nama_clean_nasabah, nama_teroris)

            if sim > best_score:
                best_score = sim

            if sim >= THRESHOLD:
                is_match = True
                original_name = teroris['all_names'][i]
                if original_name not in matched_alias_for_this_teroris:
                    matched_alias_for_this_teroris.append(original_name)

        if is_match:
            match_found = True
            matched_teroris_data = teroris['row_data']
            for alias in matched_alias_for_this_teroris:
                if alias not in matched_aliases:
                    matched_aliases.append(alias)

    # Jika match ditemukan
    if match_found and matched_teroris_data is not None:
        result_row = {}

        # Data nasabah (kiri)
        result_row['CIF'] = nasabah_row[col_cif] if col_cif and col_cif in nasabah_row else ''
        result_row['Nama_Nasabah'] = nasabah_row[col_nama_nasabah] if col_nama_nasabah in nasabah_row else ''
        result_row['Tgl_Lahir_Nasabah'] = nasabah_row[col_tgl_lahir] if col_tgl_lahir and col_tgl_lahir in nasabah_row else ''
        result_row['NIK'] = nasabah_row[col_nik] if col_nik and col_nik in nasabah_row else ''

        # Data teroris (kanan) - semua kolom
        for col in df_teroris.columns:
            if col in matched_teroris_data:
                result_row[f'Teroris_{col}'] = matched_teroris_data[col]
            else:
                result_row[f'Teroris_{col}'] = ''

        # Keterangan alias yang match
        if matched_aliases:
            result_row['Keterangan_Match'] = f"Match dengan: {', '.join(matched_aliases)}"
        else:
            result_row['Keterangan_Match'] = 'Match dengan nama asli'

        results.append(result_row)

    # False Positive (70-94%)
    elif best_score >= 0.70 and best_score < THRESHOLD:
        fp_row = {
            'CIF': nasabah_row[col_cif] if col_cif and col_cif in nasabah_row else '',
            'Nama_Nasabah': nasabah_row[col_nama_nasabah] if col_nama_nasabah in nasabah_row else '',
            'Tgl_Lahir_Nasabah': nasabah_row[col_tgl_lahir] if col_tgl_lahir and col_tgl_lahir in nasabah_row else '',
            'NIK': nasabah_row[col_nik] if col_nik and col_nik in nasabah_row else '',
            'Similarity_Tertinggi': round(best_score, 4),
            'Keterangan': f'Potensi False Positive ({best_score:.2%}) - Perlu Verifikasi'
        }
        false_positives.append(fp_row)

    # False Negative (50-70%)
    elif best_score >= 0.50 and best_score < 0.70:
        fn_row = {
            'CIF': nasabah_row[col_cif] if col_cif and col_cif in nasabah_row else '',
            'Nama_Nasabah': nasabah_row[col_nama_nasabah] if col_nama_nasabah in nasabah_row else '',
            'Tgl_Lahir_Nasabah': nasabah_row[col_tgl_lahir] if col_tgl_lahir and col_tgl_lahir in nasabah_row else '',
            'NIK': nasabah_row[col_nik] if col_nik and col_nik in nasabah_row else '',
            'Similarity_Tertinggi': round(best_score, 4),
            'Keterangan': f'Potensi False Negative ({best_score:.2%}) - Mungkin Terlewat'
        }
        false_negatives.append(fn_row)

    # Progress
    if (idx + 1) % 100 == 0:
        print(f"   Proses: {idx+1}/{total_nasabah}")

print(f"\n✅ PROSES SELESAI!")
print(f"   • Total Match: {len(results)}")
print(f"   • False Positive: {len(false_positives)}")
print(f"   • False Negative: {len(false_negatives)}")

# ================================================================
# STEP 6: BUAT DATAFRAME
# ================================================================

print("\n📊 STEP 6: MEMBUAT DATAFRAME")
print("-"*70)

df_results = pd.DataFrame(results)
df_fp = pd.DataFrame(false_positives)
df_fn = pd.DataFrame(false_negatives)

# ================================================================
# STEP 7: EXPORT KE EXCEL
# ================================================================

print("\n💾 STEP 7: EXPORT KE EXCEL")
print("-"*70)

output_file = 'HASIL_MATCHING_TERORIS.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Sheet 1: Match
    if len(df_results) > 0:
        df_results.to_excel(writer, sheet_name='MATCH', index=False)
        print(f"✅ Sheet 'MATCH': {len(df_results)} data")
    else:
        pd.DataFrame({'Info': ['TIDAK ADA NASABAH YANG MATCH DENGAN DATA TERORIS']}).to_excel(
            writer, sheet_name='MATCH', index=False
        )

    # Sheet 2: False Positive
    if len(df_fp) > 0:
        df_fp.to_excel(writer, sheet_name='FALSE_POSITIVE', index=False)
        print(f"✅ Sheet 'FALSE_POSITIVE': {len(df_fp)} data")
    else:
        pd.DataFrame({'Info': ['TIDAK ADA FALSE POSITIVE']}).to_excel(
            writer, sheet_name='FALSE_POSITIVE', index=False
        )

    # Sheet 3: False Negative
    if len(df_fn) > 0:
        df_fn.to_excel(writer, sheet_name='FALSE_NEGATIVE', index=False)
        print(f"✅ Sheet 'FALSE_NEGATIVE': {len(df_fn)} data")
    else:
        pd.DataFrame({'Info': ['TIDAK ADA FALSE NEGATIVE']}).to_excel(
            writer, sheet_name='FALSE_NEGATIVE', index=False
        )

    # Sheet 4: Summary
    summary_data = {
        'Metrik': [
            'Total Nasabah',
            'Total Teroris',
            'Threshold Similarity',
            'Total Match Ditemukan',
            'False Positive (70-94%)',
            'False Negative (50-70%)'
        ],
        'Nilai': [
            len(df_nasabah),
            len(df_teroris),
            '95%',
            len(df_results),
            len(df_fp),
            len(df_fn)
        ]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='SUMMARY', index=False)
    print("✅ Sheet 'SUMMARY'")

# Format Excel
try:
    wb = load_workbook(output_file)
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        ws.freeze_panes = 'A2'

        for column in ws.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            adjusted_width = min(max_length + 2, 50)
            ws.column_dimensions[column_letter].width = adjusted_width

        header_fill = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
        header_font = Font(color="FFFFFF", bold=True)
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")

    wb.save(output_file)
    print("✅ Format Excel selesai")
except Exception as e:
    print(f"⚠️ Format styling: {e}")

# ================================================================
# STEP 8: DOWNLOAD
# ================================================================

print("\n📥 STEP 8: DOWNLOAD FILE")
print("-"*70)
files.download(output_file)

# ================================================================
# STEP 9: PREVIEW
# ================================================================

print("\n📋 STEP 9: PREVIEW HASIL")
print("="*70)

if len(df_results) > 0:
    print(f"\n🔴 TOTAL MATCH: {len(df_results)} nasabah")
    print("\n📌 5 DATA PERTAMA:")
    print("-"*70)

    preview_cols = ['CIF', 'Nama_Nasabah']
    if 'Keterangan_Match' in df_results.columns:
        preview_cols.append('Keterangan_Match')

    existing_cols = [col for col in preview_cols if col in df_results.columns]
    if existing_cols:
        print(df_results[existing_cols].head(5).to_string())
else:
    print("\n✅ TIDAK ADA NASABAH YANG MATCH")

# ================================================================
# STEP 10: FINAL SUMMARY
# ================================================================

print("\n" + "="*70)
print("✅ PROSES SELESAI!")
print("="*70)
print(f"\n📁 FILE: {output_file}")
print("\n📑 SHEETS:")
print("   • MATCH           → Data match (95%+)")
print("   • FALSE_POSITIVE  → Perlu verifikasi (70-94%)")
print("   • FALSE_NEGATIVE  → Mungkin terlewat (50-70%)")
print("   • SUMMARY         → Statistik")
print("\n" + "="*70)
print("✨ FILE SUDAH DI-DOWNLOAD!")
print("="*70)

🔍 SISTEM MATCHING NASABAH VS DATA TERORIS (FUZZY 95%)

📌 SISTEM AKAN MENDETEKSI KOLOM SECARA OTOMATIS

📤 STEP 1: UPLOAD FILE
----------------------------------------------------------------------

📤 UPLOAD FILE DATABASE NASABAH
   (Harus ada kolom: CIF, Nama, Tgl Lahir, NIK)


Saving Data Nasabah.xlsx to Data Nasabah.xlsx

📤 UPLOAD FILE DATABASE TERORIS
   (Harus ada kolom: Nama, Nama1-Nama31, Deskripsi, Terduga, Kode Densus, Tempat Lahir, Tanggal Lahir, WN/Asal Negara, Alamat)


Saving DTTOT.xlsx to DTTOT.xlsx

✅ File nasabah: Data Nasabah.xlsx
✅ File teroris: DTTOT.xlsx

📂 STEP 2: MEMBACA DATA
----------------------------------------------------------------------

✅ Data Nasabah: 33273 baris, 4 kolom
✅ Data Teroris: 531 baris, 39 kolom

🔍 STEP 3: AUTO DETECT KOLOM
----------------------------------------------------------------------

📌 DETEKSI KOLOM NASABAH:

📌 DETEKSI KOLOM TERORIS:

✅ KOLOM YANG TERDETEKSI:
   • CIF Nasabah          : CIF
   • Nama Nasabah         : Nama
   • Tgl Lahir Nasabah    : Tgl Lahir
   • NIK Nasabah          : NIK
   • Nama Asli Teroris    : Nama
   • Kolom Alias Teroris  : 31 kolom ditemukan

   📋 ALIAS YANG DITEMUKAN:
      1. Nama1
      2. Nama2
      3. Nama3
      4. Nama4
      5. Nama5
      6. Nama6
      7. Nama7
      8. Nama8
      9. Nama9
      10. Nama10
      ... dan 21 kolom lainnya

⚙️ STEP 4: PROSES MATCHING
----------------------------------------------------------------------

🔄 MEMPROSES MATCHING (95% thresho

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📋 STEP 9: PREVIEW HASIL

🔴 TOTAL MATCH: 203 nasabah

📌 5 DATA PERTAMA:
----------------------------------------------------------------------
      CIF  Nama_Nasabah            Keterangan_Match
0  103553  ABDUL RAHMAN  Match dengan: ABDUL RAHMAN
1  103554  ABDUL ROHMAN  Match dengan: ABDUL ROHMAN
2  103772   SITI AISYAH   Match dengan: SITI AISYAH
3  106606    ZULKARNAIN    Match dengan: ZULKARNAIN
4  204057       IBRAHIM   Match dengan: Dr. IBRAHIM

✅ PROSES SELESAI!

📁 FILE: HASIL_MATCHING_TERORIS.xlsx

📑 SHEETS:
   • MATCH           → Data match (95%+)
   • FALSE_POSITIVE  → Perlu verifikasi (70-94%)
   • FALSE_NEGATIVE  → Mungkin terlewat (50-70%)
   • SUMMARY         → Statistik

✨ FILE SUDAH DI-DOWNLOAD!
